# Smoke test

Query the new Aura DB after bulk_reingest finishes.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG

driver = GraphDatabase.driver(
    os.environ['NEO4J_NEW_URI'],
    auth=(os.environ['NEO4J_NEW_USERNAME'], os.environ['NEO4J_NEW_PASSWORD'])
)
DB = os.environ.get('NEO4J_NEW_DATABASE', 'neo4j')
embedder = OpenAIEmbeddings(model='text-embedding-3-small')

## Pattern 1: Pure vector retrieval

In [ ]:
vr = VectorRetriever(driver=driver, index_name='chunk_embedding', embedder=embedder, neo4j_database=DB)
for item in vr.search(query_text='wrong-way risk in XVA close-out', top_k=5).items:
    print(item.content[:200], '...')
    print('  score:', item.metadata.get('score'))
    print()

## Pattern 2: Vector + Cypher filter (scoped to topic)

In [ ]:
rq = '''
MATCH (chunk)<-[:HAS_CHUNK]-(d:Document)-[:FROM_DOCUMENT]->(p:Paper)
OPTIONAL MATCH (p)-[:IN_TOPIC]->(t:Topic)
WHERE t.name = $topic
RETURN chunk.text AS text, p.id AS paper_id, p.title AS title
'''
vcr = VectorCypherRetriever(driver=driver, index_name='chunk_embedding', retrieval_query=rq, embedder=embedder, neo4j_database=DB)
for item in vcr.search(query_text='exponential change of measure', query_params={'topic': 'XVA'}, top_k=5).items:
    print(item.content[:200], '...')
    print()

## Pattern 3: Pure Cypher graph traversal

In [ ]:
with driver.session(database=DB) as s:
    for r in s.run('''
        MATCH (p:Paper)-[:USES]->(c:Concept)
        RETURN c.name AS concept, count(DISTINCT p) AS papers
        ORDER BY papers DESC LIMIT 10
    '''):
        print(r['papers'], r['concept'])

## Pattern 4: Full GraphRAG (retrieval + LLM synthesis)

In [ ]:
llm = OpenAILLM(model_name='gpt-4o-mini')
rag = GraphRAG(retriever=vr, llm=llm)
resp = rag.search(
    query_text='How does the close-out convention affect XVA pricing under counterparty default?',
    return_context=True,
)
print(resp.answer)